## 👨‍💼 Business Scenario

You are working as a Data Analyst at an e-commerce company.

The Operations Manager proposes:

> "We are considering investing in faster delivery. Before spending more money, I want to know whether faster delivery could improve customer satisfaction."
> 

Your task is to use historical order data to investigate the relationship and then design a proper A/B experiment.

In [5]:
import pandas as pd
import seaborn as sns

In [6]:
cd = pd.read_csv("olist_customers_dataset.csv") 
gd=pd.read_csv("olist_geolocation_dataset.csv")
ot=pd.read_csv("olist_order_items_dataset.csv")
od=pd.read_csv("olist_orders_dataset.csv")
pyd=pd.read_csv("olist_order_payments_dataset.csv")
ord=pd.read_csv("olist_order_reviews_dataset.csv")
odd=pd.read_csv("olist_orders_dataset.csv")
prd=pd.read_csv("olist_products_dataset.csv")
osd=pd.read_csv("olist_sellers_dataset.csv")
pcn=pd.read_csv("product_category_name_translation.csv")

In [7]:
od

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [8]:
od['order_delivered_customer_date']=pd.to_datetime(od['order_delivered_customer_date'])
od['order_purchase_timestamp']=pd.to_datetime(od['order_purchase_timestamp'])

In [9]:
od['actual_delivery_time']=(od['order_delivered_customer_date']-od['order_purchase_timestamp']).dt.total_seconds() / (24 * 60 * 60)

In [10]:
ord.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [11]:
data_relation=od[['order_id','actual_delivery_time']].merge(ord[['order_id','review_score']],on='order_id',how='left')

In [12]:
data_relation['actual_delivery_time']=(data_relation['actual_delivery_time'].round(0))

In [13]:
data_relation

,order_id,actual_delivery_time,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,14.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,3.0,5.0
...,...,...,...
99987,9c5dedf39a927c1b2549525ed64a053c,8.0,5.0
99988,63943bddc261676b46f01ca7ac2f7bd8,22.0,4.0
99989,83c1379a015df1e13d02aae0204711ab,25.0,5.0
99990,11c177c8e97725db2631073c19f07b62,17.0,2.0


In [14]:
data_relation.describe()

,actual_delivery_time,review_score
count,97005.000000,99224.000000
mean,12.523385,4.086421
std,9.546804,1.347579
min,1.000000,1.000000
25%,7.000000,4.000000
50%,10.000000,5.000000
75%,16.000000,5.000000
max,210.000000,5.000000


In [15]:
arv=data_relation.groupby('actual_delivery_time')['review_score'].mean().reset_index()

In [16]:
data_relation.describe()

,actual_delivery_time,review_score
count,97005.000000,99224.000000
mean,12.523385,4.086421
std,9.546804,1.347579
min,1.000000,1.000000
25%,7.000000,4.000000
50%,10.000000,5.000000
75%,16.000000,5.000000
max,210.000000,5.000000


In [17]:
data_relation.duplicated().sum()

np.int64(349)

In [18]:
data_relation.drop_duplicates(inplace=True)

In [19]:
data_relation.isnull().sum()

order_id                   0
actual_delivery_time    2978
review_score             768
dtype: int64

In [20]:
data_relation.dropna(inplace=True)

In [21]:
data_relation.describe()

,actual_delivery_time,review_score
count,96019.000000,96019.000000
mean,12.480759,4.154553
std,9.465961,1.285491
min,1.000000,1.000000
25%,7.000000,4.000000
50%,10.000000,5.000000
75%,16.000000,5.000000
max,208.000000,5.000000


In [22]:
data_relation

,order_id,actual_delivery_time,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,14.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,3.0,5.0
...,...,...,...
99987,9c5dedf39a927c1b2549525ed64a053c,8.0,5.0
99988,63943bddc261676b46f01ca7ac2f7bd8,22.0,4.0
99989,83c1379a015df1e13d02aae0204711ab,25.0,5.0
99990,11c177c8e97725db2631073c19f07b62,17.0,2.0


In [23]:
data_relation.groupby('review_score')['actual_delivery_time'].mean().reset_index()

,review_score,actual_delivery_time
0,1.0,21.264066
1,2.0,16.609939
2,3.0,14.224097
3,4.0,12.268490
4,5.0,10.643432


In [24]:
data_relation['actual_delivery_time'].median()

10.0

### Group A — Slower Delivery

Orders with delivery time greater than the median delivery time.

### Group B — Faster Delivery

Orders with delivery time less than or equal to the median delivery time.

In [25]:
group_A=data_relation[data_relation['actual_delivery_time']>data_relation['actual_delivery_time'].median()]

In [26]:
group_B=data_relation[data_relation['actual_delivery_time']<=data_relation['actual_delivery_time'].median()]

In [27]:
#historical comparison groups
median_delivery = data_relation["actual_delivery_time"].median()
print("Median:", median_delivery)
print("Group A:", len(group_A))
print("Group B:", len(group_B))

Median: 10.0
Group A: 46411
Group B: 49608


In [28]:
print(group_A["actual_delivery_time"].min())
print(group_B["actual_delivery_time"].max())

11.0
10.0


In [29]:
group_sizes = pd.DataFrame({
    "Group": ["Faster Delivery", "Slower Delivery"],
    "Orders": [len(group_B), len(group_A)]
})

group_sizes

,Group,Orders
0,Faster Delivery,49608
1,Slower Delivery,46411


In [30]:
faster =len(group_B)
slower =len(group_A)

In [31]:
both_sum=len(group_A)+len(group_B)
print('Difference in group sizes:', abs(faster-slower))
print('Faster %:', (faster/both_sum)*100)
print('Slower %:', (slower/both_sum)*100)


Difference in group sizes: 3197
Faster %: 51.66477468001125
Slower %: 48.33522531998875


The two groups are reasonably large and have very similar sample sizes. This is expected because the median was used to divide the orders into two groups. Both groups contain enough observations for further comparison.

In [32]:
faster=group_B['order_id'].nunique()
slower=group_A['order_id'].nunique()

In [33]:
avg_review_score_B=group_B['review_score'].mean().round(0)
avg_review_score_A=group_A['review_score'].mean().round(0)

In [34]:
median_review_score_B=group_B['review_score'].median()
median_review_score_A=group_A['review_score'].median()

In [35]:
data_5star_reviews=data_relation[data_relation['review_score']==5]

In [36]:
group_A_5star = len(data_5star_reviews[data_5star_reviews['actual_delivery_time']>median_delivery])
group_B_5star = len(data_5star_reviews[data_5star_reviews['actual_delivery_time']<=median_delivery])

In [37]:
group_B_low_rating = len(group_B[group_B['review_score']==group_B['review_score'].min()])
group_A_low_rating = len(group_A[group_A['review_score']==group_A['review_score'].min()])


In [38]:
Metrics = pd.DataFrame({
    "Metric": [
        "Average Review Score",
        "Median Review Score",
        "5-Star Reviews %",
        "Low Rating Reviews %"
    ],
    "Group A": [
        avg_review_score_A,
        median_review_score_A,
        round(group_A_5star / len(group_A) * 100, 2),
        round(group_A_low_rating / len(group_A) * 100, 2)
    ],
    "Group B": [
        avg_review_score_B,
        median_review_score_B,
        round(group_B_5star / len(group_B) * 100, 2),
        round(group_B_low_rating / len(group_B) * 100, 2)
    ]
})


In [39]:
Metrics

,Metric,Group A,Group B
0,Average Review Score,4.00,4.00
1,Median Review Score,5.00,5.00
2,5-Star Reviews %,51.24,66.59
3,Low Rating Reviews %,14.10,5.73



## Difference of both group reviews

In [40]:
difference=group_B['review_score'].mean() - group_A['review_score'].mean()

In [41]:
print(difference.round(2))

0.47


A 0.40-point improvement on a 5-point rating scale is potentially meaningful. Customers in the faster-delivery group gave ratings that were, on average, 0.40 points higher than customers in the slower-delivery group.

Is a 0.40-point increase in average rating large enough to justify investing in faster delivery?

Faster delivery is associated with a 0.40-point higher average review score. This appears practically meaningful, but further analysis is needed to determine whether the difference is statistically reliable and whether faster delivery itself is responsible for the improvement.

# hypothesis testing

### Null Hypothesis — H₀

Faster delivery is not associated with higher customer review scores.

### Alternative Hypothesis — H₁

Faster delivery is associated with higher customer review scores.

              YOUR BUSINESS QUESTION
                       ↓
       "Do faster deliveries improve ratings?"
                       ↓
                 H₀ and H₁
                       ↓
             Group A vs Group B
                       ↓
                    T-TEST
                       ↓
              ┌───────────────┐
              │               │
         T-statistic       P-value
              │               │
       How big is the      How surprising
        difference?         is the result
              │              under H₀?
              │               │
              └───────┬───────┘
                      ↓
               Compare p to 0.05
                      ↓
             ┌────────┴────────┐
             ↓                 ↓
         p < 0.05          p ≥ 0.05
             ↓                 ↓
        Reject H₀        Fail to reject H₀

In [49]:
print("Group A Mean:", group_A["review_score"].mean())
print("Group B Mean:", group_B["review_score"].mean())

Group A Mean: 3.9120682596798173
Group B Mean: 4.381410256410256


In [53]:
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(group_A['review_score'], group_B['review_score'], equal_var=False)
print("T-statistic:", t_stat)
print("P-value:", p_value) 
# P-value is extremely small, that's why it shows 0.
# It doesn't mean the actual p-value is exactly 0;
# it is just an extremely small number.

T-statistic: -57.007102966457566
P-value: 0.0


In [55]:
mean_a = group_A["review_score"].mean()
mean_b = group_B["review_score"].mean()

print("Group A Mean:", mean_a)
print("Group B Mean:", mean_b)
print("Difference:", mean_b - mean_a)

Group A Mean: 3.9120682596798173
Group B Mean: 4.381410256410256
Difference: 0.4693419967304391


In [58]:
#Conclusion
#T-statistic = -57.0071 → very large difference relative to the variation,
#P-value ≈ extremely close to 0 → very strong statistical evidence against H₀,
#Since p < 0.05, you reject H₀

We reject the null hypothesis. There is statistically significant evidence of a difference in review scores between the faster- and slower-delivery groups. The faster-delivery group has the higher average review score, which is consistent with our alternative hypothesis.

In [65]:
import numpy as np

n_A = len(group_A["review_score"])
n_B = len(group_B["review_score"])

std_A = group_A["review_score"].std()
std_B = group_B["review_score"].std()

se = np.sqrt(
    (std_A**2 / n_A) +
    (std_B**2 / n_B)
)

print("Standard Error:", se)
mean_a = group_A["review_score"].mean()
mean_b = group_B["review_score"].mean()
Difference = mean_b - mean_a
print("Difference:", Difference)
print((se/Difference).round(2)*100)


Standard Error: 0.008233044170067925
Difference: 0.4693419967304391
2.0


In [76]:
from scipy.stats import ttest_ind

# Welch's t-test
result = ttest_ind(
    group_B["review_score"],
    group_A["review_score"],
    equal_var=False
)

# Results
print("T-statistic:", result.statistic)
print("P-value:", result.pvalue)

# 95% Confidence Interval
ci = result.confidence_interval(confidence_level=0.95)

print("95% CI:", ci.low, "to", ci.high)

T-statistic: 57.007102966457566
P-value: 0.0
95% CI: 0.45320530197088216 to 0.485478691489996


The t-test gave a t-statistic of 57.01 and p-value ≈ 0. The 95% CI shows the difference between the two averages is 0.45–0.49.